In [2]:
import struct

def sha256_full(message):
    K = [0x428a2f98, 0x71374491, 0xb5c0fbcf, 0xe9b5dba5, 0x3956c25b, 0x59f111f1, 0x923f82a4, 0xab1c5ed5, 0xd807aa98, 0x12835b01, 0x243185be, 0x550c7dc3, 0x72be5d74, 0x80deb1fe, 0x9bdc06a7, 0xc19bf174, 0xe49b69c1, 0xefbe4786, 0x0fc19dc6, 0x240ca1cc, 0x2de92c6f, 0x4a7484aa, 0x5cb0a9dc, 0x76f988da, 0x983e5152, 0xa831c66d, 0xb00327c8, 0xbf597fc7, 0xc6e00bf3, 0xd5a79147, 0x06ca6351, 0x14292967, 0x27b70a85, 0x2e1b2138, 0x4d2c6dfc, 0x53380d13, 0x650a7354, 0x766a0abb, 0x81c2c92e, 0x92722c85, 0xa2bfe8a1, 0xa81a664b, 0xc24b8b70, 0xc76c51a3, 0xd192e819, 0xd6990624, 0xf40e3585, 0x106aa070, 0x19a4c116, 0x1e376c08, 0x2748774c, 0x34b0bcb5, 0x391c0cb3, 0x4ed8aa4a, 0x5b9cca4f, 0x682e6ff3, 0x748f82ee, 0x78a5636f, 0x84c87814, 0x8cc70208, 0x90befffa, 0xa4506ceb, 0xbef9a3f7, 0xc67178f2]
    H0 = [0x6a09e667, 0xbb67ae85, 0x3c6ef372, 0xa54ff53a, 0x510e527f, 0x9b05688c, 0x1f83d9ab, 0x5be0cd19]
    
    def rotr(x, n): return ((x >> n) | (x << (32 - n))) & 0xffffffff
    def ch(x, y, z): return (x & y) ^ (~x & z)
    def maj(x, y, z): return (x & y) ^ (x & z) ^ (y & z)
    def ep0(x): return rotr(x, 2) ^ rotr(x, 13) ^ rotr(x, 22)
    def ep1(x): return rotr(x, 6) ^ rotr(x, 11) ^ rotr(x, 25)
    def sig0(x): return rotr(x, 7) ^ rotr(x, 18) ^ (x >> 3)
    def sig1(x): return rotr(x, 17) ^ rotr(x, 19) ^ (x >> 10)
    
    if isinstance(message, str): message = message.encode('utf-8')
    original_len = len(message) * 8
    message += b'\x80'
    while (len(message) * 8 + 64) % 512 != 0: message += b'\x00'
    message += struct.pack('>Q', original_len)
    
    block = message[0:64]
    W = [0] * 64
    for t in range(64):
        if t < 16: W[t] = struct.unpack('>I', block[t*4:(t+1)*4])[0]
        else: W[t] = (sig1(W[t-2]) + W[t-7] + sig0(W[t-15]) + W[t-16]) & 0xffffffff
    
    a, b, c, d, e, f, g, h = H0[0], H0[1], H0[2], H0[3], H0[4], H0[5], H0[6], H0[7]
    
    round_data = []
    for t in range(64):
        T1 = (h + ep1(e) + ch(e, f, g) + K[t] + W[t]) & 0xffffffff
        T2 = (ep0(a) + maj(a, b, c)) & 0xffffffff
        round_data.append((t, T1, T2, W[t], a, b, c, d, e, f, g, h))
        h = g; g = f; f = e; e = (d + T1) & 0xffffffff; d = c; c = b; b = a; a = (T1 + T2) & 0xffffffff
    
    final_vars = [a, b, c, d, e, f, g, h]
    H_final = [(H0[i] + final_vars[i]) & 0xffffffff for i in range(8)]
    
    return W, H0, H_final, final_vars, round_data

W, H0, H_final, final_vars, round_data = sha256_full("abc")

print("FORWARD_EXTRACTED")
print("W0-W15:", " ".join([f"0x{w:08x}" for w in W[:16]]))
print("H0-H7:", " ".join([f"0x{h:08x}" for h in H0]))
print()
print("FIRST_VALUE_TO_FIND")
print("T1_63 = 0xa8467f25")
print()
print("ALL_VALUES_TO_FIND (Round 63 down to 0)")
print("Round|T1|T2|W")
print("-"*40)
for t in range(63, -1, -1):
    r = round_data[t]
    print(f"{r[0]:2d}|0x{r[1]:08x}|0x{r[2]:08x}|0x{r[3]:08x}")

FORWARD_EXTRACTED
W0-W15: 0x61626380 0x00000000 0x00000000 0x00000000 0x00000000 0x00000000 0x00000000 0x00000000 0x00000000 0x00000000 0x00000000 0x00000000 0x00000000 0x00000000 0x00000000 0x00000018
H0-H7: 0x6a09e667 0xbb67ae85 0x3c6ef372 0xa54ff53a 0x510e527f 0x9b05688c 0x1f83d9ab 0x5be0cd19

FIRST_VALUE_TO_FIND
T1_63 = 0xa8467f25

ALL_VALUES_TO_FIND (Round 63 down to 0)
Round|T1|T2|W
----------------------------------------
63|0xa8467f25|0xa827b133|0x12b1edeb
62|0xfb5b0d9e|0xd83f13c7|0xeeaba2cc
61|0xd42a5147|0x30a7fc25|0x668b2ff8
60|0x994dc019|0x1f106cd0|0xa43fcf15
59|0x794f142a|0x3d5f7bd5|0x78bc8d4b
58|0x2ebf62eb|0xd0f7a187|0x9fe3095e
57|0x88d3d0b3|0x378f03bc|0xebe6b238
56|0x8b3ebddd|0x7192ca9e|0xef57b9cd
55|0xcc5cac49|0x6c6fecca|0xb20f7a99
54|0x6b0912ae|0xd3bb4a2d|0x1487472c
53|0xae2b8d1b|0x4744a2c0|0xc21462bc
52|0x4d290015|0xc2f6d27a|0x84badedd
51|0x07b0e991|0x80bd9091|0xa9993667
50|0xb1a5b847|0xc85f505a|0xb9e66c34
49|0xfbb0ed7b|0xf56b0d2d|0xcc7617db
48|0x970eb823|0x6e66c4c8|0x

12b1edeba827b133a8467f25eeaba2ccd83f13c7fb5b0d9e668b2ff830a7fc25
d42a5147a43fcf151f106cd0994dc01978bc8d4b3d5f7bd5794f142a9fe3095e
d0f7a1872ebf62ebebe6b238378f03bc88d3d0b3ef57b9cd7192ca9e8b3ebddd
b20f7a996c6feccacc5cac491487472cd3bb4a2d6b0912aec21462bc4744a2c0
ae2b8d1b84badeddc2f6d27a4d290015a999366780bd909107b0e991b9e66c34
c85f505ab1a5b847cc7617dbf56b0d2dfbb0ed7bfb3e89cb6e66c4c8970eb823
065c43dadfeed65a61835c337a290d5de37a8e05bdd186ab840abf279a7c531f
d82ef8720c4763f254ba7d8bd1a80a8a27333ba31993cbdf83b7e3b43e246a79
a6a241737107cc8bf0a64f5a909a54224eac110d9f47bf94f1d836b4e48ed0b2
24641522c08f77beb95af0bc9409e33e26f4d24e46d0a83c72af830a8e37bebc
a6a757480a8b3996d314df4d6e917d64f10a5c62e4a4090380fcc6e1aff4ffc1
8be1657b1dc60e113b68ba734541bb52b91e92a393f5997f323bb1da66a5732d
d3b7973b312da00b42859bea702138a49fbe43a44adae67eec8726cbc0067a76
dfff90ba9d209d67f14e074a44e67b7f32663c5b4fa8f2bd282a826be5bc3909
9685c1e51fa902d7b73679a27c10335c51d7d021c8215c1a91cec40a8a5d642e
e2e2c38efd7f8679c855b71412dcbfdb5c25ccb44b7d958b0183fc00a436cdce
1e299f9f3e9d7b7834c69cce68ffee95600003c6e288ae48ff695deb7da86405
f252423c0c253983000f000017bfe093ab3bf93e616263802937a18ff8a2f90c
00000018a0c0db6a1039482400000000fc6a11c6c3fa4e1800000000f15fd6a3
e7d3147300000000f73e94319052955f00000000fb506a1ffbcf5b8a00000000
e930838c5e682068000000009b0018caf1871ba1000000005f7b59972e899322
000000004e8bf74c3714841300000000129ea726d2645c5a00000000146a82f5
16d78700000000001dfde3f2e642b678000000001a7ad47dbad621e900000000
8b01bc413dc18b66000000001e0b53963c5f86176162638008909ae554da50e8


The numeric string 313650665 appears at the 1,856,365,906th decimal digit of Pi.
The numeric string 313650666 appears at the 744,232,464th decimal digit of Pi.
The numeric string 313650667 appears at the 1,139,731,283rd decimal digit of Pi   -> our digits
The numeric string 313650668 appears at the 1,131,662,640 decimal digit of Pi.
The numeric string 313650669 appears at the 1,435,671,942nd decimal digit of Pi.
The numeric string 313650670 appears at the 689,654,818th decimal digit of Pi.






The numeric string 313650665 appears at the 376,312,397th decimal digit of E.
The numeric string 313650666 appears at the 891,751,152nd decimal digit of E.
The numeric string 313650667 appears at the 140,846,927th decimal digit of E   -> our digits
The numeric string 313650668 was not found in the first 2,000,000,000 decimal digits of E.
The numeric string 313650669 appears at the 628,697,573rd decimal digit of E.
The search string "313650670" was not found in the first 2,000,000,000 decimal digits of E.





The numeric string 313650665 appears at the 1,274,751,777th decimal digit of the Square Root of 2.
The numeric string 313650666 appears at the 1,377,967,996th decimal digit of the Square Root of 2.
The numeric string 313650667 appears at the 644,044,488th decimal digit of the Square Root of 2.-> our digits
The search string "313650668" was not found in the first 2,000,000,000 decimal digits of the Square Root of 2.
The search string "313650669" was not found in the first 2,000,000,000 decimal digits of the Square Root of 2.
The numeric string 313650670 appears at the 47,878,446th decimal digit of the Square Root of 2.




The numeric string 313650665 appears at the 265,904,557th decimal digit of the Golden Ratio (Phi).
The numeric string 313650666 appears at the 397,289,304th decimal digit of the Golden Ratio (Phi).
The search string "313650667" was not found in the first 500,000,000 decimal digits of the Golden Ratio (Phi).-> our digits
The search string "313650668" was not found in the first 500,000,000 decimal digits of the Golden Ratio (Phi).
The search string "313650669" was not found in the first 500,000,000 decimal digits of the Golden Ratio (Phi).
The numeric string 313650670 appears at the 303,595,130 decimal digit of the Golden Ratio (Phi).





25 decimal digits of Pi starting from position 313,650,667:
 
3683296543410382829931628



25 decimal digits of E starting from position 313,650,667:
 
9782649924475662563964248


25 decimal digits of the Square Root of 2 starting from position 313,650,667:
 
0744846307572025925730678


25 decimal digits of the Golden Ratio (Phi) starting from position 313,650,667:
 
1356631853302406293453125








